# Point review: live hard coral errors, image by image

Review the model's live hard coral errors at the annotated CoralNet points, in the notebook (no server, nothing to
forward). Same 10 images as `Coral_Cover_Qualitative_Live_10src.ipynb` (`cbm_scratch100k_base_taxv2`, step 100k,
P(scleractinia) × P(live) ≥ 0.5 vs annotated live hard coral).

Each point gets an automatic outcome: **TP / TN** (model and annotation agree), **FP** (model says live hard coral,
annotation says not), **FN** (the reverse), **NA** (annotation not given). Step through the FP / FN points and give each a
verdict; points you don't touch keep their automatic outcome.

| key | action |
|---|---|
| `→` / `Tab`, `←` / `⇧Tab` | next / previous point (moves to the next image after the last) |
| `m` | **model wrong**: the annotation is right, the error is real |
| `a` | **annotation wrong**: the model is right |
| `?` | ambiguous; `u` clears the verdict |
| `n` / `p` | next / previous image |
| `1` `2` `3` `0` | map: P(scleractinia) × P(live), P(scleractinia), P(live), none |
| `z` / `f`, wheel, drag | zoom on the point / fit; zoom and pan |
| `l`, `e` | labels on all points; cycle FP + FN / unreviewed / all points |

**Click the widget once before using the keys** (it then has keyboard focus and the notebook's own shortcuts are off).
Every verdict is saved at once to `eval_out/coral_cover/review/base_taxv2_live/reviews.csv`; re-running the widget cell
picks up where you left off.

Predictions were saved beforehand with
```
python scripts/review_export.py --target live_hard --n-per-source 1 --out eval_out/coral_cover/review/base_taxv2_live
```
(`--n-per-source 100` for the full set, into a new folder).

In [1]:
import sys
from pathlib import Path

REPO = Path("..").resolve() if Path("../scripts").exists() else Path(".").resolve()
sys.path.insert(0, str(REPO / "scripts"))
from review_widget import PointReview  # noqa: E402

DATA = REPO / "eval_out/coral_cover/review/base_taxv2_live"
review = PointReview(DATA, reviewer="viktor", height=820)
review

## Verdicts so far
Latest verdict per reviewed point (re-run after reviewing).

In [2]:
done = review.reviews_df()
print(f"{len(done)} points reviewed")
done.pivot_table(index="outcome", columns="verdict", values="idx", aggfunc="count", fill_value=0)

22 points reviewed


verdict,ambiguous,annotation_wrong,model_wrong
outcome,,,
FN,1,11,10


## Corrected cover per image

Annotated live hard coral cover before and after the review: an FP marked *annotation wrong* becomes a positive, an FN marked
*annotation wrong* becomes a negative. *Model wrong* and *ambiguous* keep the annotation.

In [3]:
import json

import pandas as pd

manifest = json.loads((DATA / "manifest.json").read_text())
flips = done[done.verdict == "annotation_wrong"]
rows = []
for m in manifest["images"]:
    pts = pd.DataFrame(json.loads((DATA / "points" / f"{m['image_id']}.json").read_text()))
    flipped = pts[pts.idx.isin(flips.loc[flips.image_id == m["image_id"], "idx"].astype(int))]
    call = flipped["call_" + manifest["target"]]
    gained = ((flipped.gt_code == 1) & call).sum()  # FP: annotation said no, the model (and reviewer) say yes
    lost = ((flipped.gt_code == 2) & ~call).sum()  # FN: annotation said yes, the model (and reviewer) say no
    n = len(pts)
    gt = (pts.gt_code == 2).sum()
    rows.append({"source": m["source_id"], "image": m["image_id"], "points": n,
                 "annotated cover": gt / n, "corrected cover": (gt + gained - lost) / n,
                 "model at points": m["model_points_cover"], "annotations flipped": int(gained + lost)})
pd.DataFrame(rows).style.format({c: "{:.0%}" for c in ("annotated cover", "corrected cover", "model at points")})

,source,image,points,annotated cover,corrected cover,model at points,annotations flipped
0,1990,1764460,25,0%,0%,0%,0
1,2955,2303865,25,28%,16%,0%,3
2,3606,3624447,25,0%,0%,0%,0
3,3697,3356966,50,50%,48%,46%,1
4,4342,3665958,64,58%,56%,36%,1
5,4957,4165702,5,0%,0%,0%,0
6,6405,5785586,10,0%,0%,0%,0
7,6551,5314208,50,12%,10%,2%,1
8,6702,5483636,13,0%,0%,0%,0
9,7116,5651052,20,25%,0%,0%,5
